<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/split_wise4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [106]:
from dataclasses import dataclass, replace, field, copy

@dataclass(eq=False)
class Person:
    name: str = field(compare=False)
    bal: int = 0

    def __radd__(self, other):
        if other == 0: return self.bal
        return self.bal + other

    def __lt__(self, other):
        return self.bal < other.bal

class Pool:
    def __init__(self, *members):
        self.pool = [*members]

    def spend(self, person, amount):
        for p in self.pool:
            p.bal -= amount / len(self.pool)
        person.bal += amount

    def __repr__(self):
        return ", ".join(f"{p.name} {p.bal}" for p in self.pool)

    def groups(self):
        creds, debts = [], []
        for p in self.pool:
            if p.bal > 0:
                creds.append(p)
            elif p.bal < 0:
                debts.append(p)
        return tuple(creds), tuple(debts)

In [107]:
a,b,c = Person("A",5), Person("B",-5), Person("C", 0)
p = Pool(a,b,c)
p.spend(c, 9)
print(p)

A 2.0, B -8.0, C 6.0


In [108]:
@dataclass
class Transaction:
    payer: Person
    payee: Person
    amount: int

    def __repr__(self):
        return f"{self.payer.name}->{self.payee.name}: {self.amount}"

import heapq

class HeapSettler:
    @staticmethod
    def settle(creds, debts):
        txns = []
        creds = [replace(p, bal=-p.bal) for p in creds]
        debts = [copy.copy(p) for p in debts]
        heapq.heapify(creds)
        heapq.heapify(debts)
        while creds and debts:
            c = heapq.heappop(creds)
            d = heapq.heappop(debts)
            b = min(-c.bal, -d.bal)
            txns.append(Transaction(d, c, b))
            if c.bal > d.bal:
                d.bal += b
                heapq.heappush(debts, d)
            elif c.bal < d.bal:
                c.bal += b
                heapq.heappush(creds, c)
        return txns

In [109]:
HeapSettler.settle(*p.groups())
p = Pool(Person("A", +12), Person("B", +11), Person("C", +10),
         Person("D", +9), Person("E", -15), Person("F", -14),
         Person("G", -8), Person("H", -5))
HeapSettler.settle(*p.groups())

[E->A: 12, F->B: 11, G->C: 8, H->D: 5, F->D: 3, E->C: 2, E->D: 1]

In [110]:
from functools import cache
from itertools import product

def partitions(s):
    for i in range(1, 2**len(s)):
        s1 = tuple(s[j] for j in range(len(s)) if 2**j & i)
        s2 = tuple(s[j] for j in range(len(s)) if not 2**j & i)
        yield s1, s2

@cache
def max0sets(creds, debts):
    max_subsets = []
    for cred, debt in product(partitions(creds), partitions(debts)):
        subsets = []
        if sum(cred[0]) + sum(debt[0]) == 0:
            subsets = [(cred[0], debt[0])]
            if not cred[1] or not cred[1]:
                pass
            elif len(cred[1]) == 1 or len(cred[1]) == 1:
                subsets += [(cred[1], debt[1])]
            else:
                subsets += max0sets(cred[1], debt[1])
            if len(subsets) > len(max_subsets):
                max_subsets = subsets
    return max_subsets

class OptimumSettler:
    @staticmethod
    def settle(creds, debts):
        print(creds, debts)
        groups = max0sets(creds, debts)
        txns = []
        for g in groups:
            txns += HeapSettler.settle(*g)
        return txns

In [111]:
p = Pool(Person("A", +12), Person("B", +11), Person("C", +10),
         Person("D", +9), Person("E", -15), Person("F", -14),
         Person("G", -8), Person("H", -5))
OptimumSettler.settle(*p.groups())

(Person(name='A', bal=12), Person(name='B', bal=11), Person(name='C', bal=10), Person(name='D', bal=9)) (Person(name='E', bal=-15), Person(name='F', bal=-14), Person(name='G', bal=-8), Person(name='H', bal=-5))


[E->A: 12, G->B: 8, E->B: 3, F->C: 10, H->D: 5, F->D: 4]

In [112]:
max0sets.cache_info()

CacheInfo(hits=0, misses=5, maxsize=None, currsize=5)